In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import Normalizer

In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

def create_advanced_features(data):
    """Create advanced feature combinations"""
    
    # Ratio Features
    data['ME_ratio'] = data['M*'] / (data['E*'] + 1e-8)
    data['VI_ratio'] = data['V*'] / (data['I*'] + 1e-8)
    data['PS_ratio'] = data['P*'] / (data['S*'] + 1e-8)
    data['ED_ratio'] = data['E*'] / (data['D*'] + 1e-8)
    data['MV_ratio'] = data['M*'] / (data['V*'] + 1e-8)
    data['EI_ratio'] = data['E*'] / (data['I*'] + 1e-8)
    data['PD_ratio'] = data['P*'] / (data['D*'] + 1e-8)
    data['SI_ratio'] = data['S*'] / (data['I*'] + 1e-8)
    
    # Product Features
    data['M_E_product'] = data['M*'] * data['E*']
    data['V_S_product'] = data['V*'] * data['S*']
    data['I_P_product'] = data['I*'] * data['P*']
    data['M_V_product'] = data['M*'] * data['V*']
    data['E_S_product'] = data['E*'] * data['S*']
    data['P_D_product'] = data['P*'] * data['D*']
    
    # Three-way Ratios
    data['MEV_ratio'] = data['M*'] / ((data['E*'] + data['V*']) + 1e-8)
    data['PIS_ratio'] = data['P*'] / ((data['I*'] + data['S*']) + 1e-8)
    data['MVD_ratio'] = data['M*'] / ((data['V*'] + data['D*']) + 1e-8)
    data['EIP_ratio'] = data['E*'] / ((data['I*'] + data['P*']) + 1e-8)
    
    # Squared and Power Features
    data['M_squared'] = data['M*'] ** 2
    data['E_squared'] = data['E*'] ** 2
    data['V_squared'] = data['V*'] ** 2
    data['S_squared'] = data['S*'] ** 2
    data['I_squared'] = data['I*'] ** 2
    
    # Square Root Features (for variance-like reduction)
    data['M_sqrt'] = np.sqrt(np.abs(data['M*']))
    data['E_sqrt'] = np.sqrt(np.abs(data['E*']))
    data['V_sqrt'] = np.sqrt(np.abs(data['V*']))
    
    # Log Features (for exponential relationships)
    data['M_log'] = np.log1p(np.abs(data['M*']))
    data['E_log'] = np.log1p(np.abs(data['E*']))
    data['V_log'] = np.log1p(np.abs(data['V*']))
    data['I_log'] = np.log1p(np.abs(data['I*']))
    
    # Interaction with Individual Features
    for col in ['E1', 'E2', 'E3', 'E4', 'E5', 'E7', 'E8', 'E9']:
        if col in data.columns:
            data[f'{col}_M_interaction'] = data[col] * data['M*']
            data[f'{col}_V_interaction'] = data[col] * data['V*']
    
    for col in ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S8']:
        if col in data.columns:
            data[f'{col}_E_interaction'] = data[col] * data['E*']
            data[f'{col}_P_interaction'] = data[col] * data['P*']
    
    # Complex Ratios
    data['MEIP_combined'] = (data['M*'] * data['E*']) / ((data['I*'] * data['P*']) + 1e-8)
    data['VSDI_combined'] = (data['V*'] * data['S*']) / ((data['D*'] * data['I*']) + 1e-8)
    data['MPVE_combined'] = (data['M*'] * data['P*']) / ((data['V*'] * data['E*']) + 1e-8)
    
    # Weighted Combinations
    data['risk_score'] = 0.3 * data['V*'] + 0.3 * data['D*'] + 0.4 * data['I*']
    data['market_strength'] = 0.4 * data['M*'] + 0.3 * data['E*'] + 0.3 * data['S*']
    data['valuation_score'] = 0.5 * data['P*'] + 0.3 * data['E*'] + 0.2 * data['M*']
    
    # Normalized Differences
    data['ME_diff_norm'] = (data['M*'] - data['E*']) / (data['M*'] + data['E*'] + 1e-8)
    data['VS_diff_norm'] = (data['V*'] - data['S*']) / (data['V*'] + data['S*'] + 1e-8)
    data['IP_diff_norm'] = (data['I*'] - data['P*']) / (data['I*'] + data['P*'] + 1e-8)
    
    # Momentum-like Features
    data['MEVS_momentum'] = data['M*'] * data['E*'] - data['V*'] * data['S*']
    data['IPVD_momentum'] = data['I*'] * data['P*'] - data['V*'] * data['D*']
    
    # Volatility Adjusted Features
    data['M_vol_adj'] = data['M*'] / (data['V*'] + 1e-8)
    data['E_vol_adj'] = data['E*'] / (data['V*'] + 1e-8)
    data['P_vol_adj'] = data['P*'] / (data['V*'] + 1e-8)
    data['S_vol_adj'] = data['S*'] / (data['V*'] + 1e-8)
    
    # Aggregate Stats across groups
    data['total_signal'] = data['M*'] + data['E*'] + data['I*'] + data['P*']
    data['total_risk'] = data['V*'] + data['D*']
    data['signal_to_risk'] = data['total_signal'] / (data['total_risk'] + 1e-8)
    
    # Cross-category features
    data['macro_tech_ratio'] = data['E*'] / (data['M*'] + 1e-8)
    data['sentiment_vol_ratio'] = data['S*'] / (data['V*'] + 1e-8)
    data['price_interest_ratio'] = data['P*'] / (data['I*'] + 1e-8)
    
    # Polynomial combinations (2nd order)
    data['M_E_poly'] = data['M*'] * data['E*'] - 0.5 * (data['M*']**2 + data['E*']**2)
    data['V_S_poly'] = data['V*'] * data['S*'] - 0.5 * (data['V*']**2 + data['S*']**2)
    
    # Standardized scores
    data['MEVIPS_mean'] = (data['M*'] + data['E*'] + data['V*'] + data['I*'] + data['P*'] + data['S*']) / 6
    data['MEVIPS_std'] = np.sqrt(
        ((data['M*'] - data['MEVIPS_mean'])**2 + 
         (data['E*'] - data['MEVIPS_mean'])**2 + 
         (data['V*'] - data['MEVIPS_mean'])**2 + 
         (data['I*'] - data['MEVIPS_mean'])**2 + 
         (data['P*'] - data['MEVIPS_mean'])**2 + 
         (data['S*'] - data['MEVIPS_mean'])**2) / 6
    )
    
    return data


def preprocessing(data, typ, scaler=None, imputer=None):
    data = data.dropna()
    
    main_features = ['E1', 'E2', 'E3', 'E4', 'E5', 'E7', 'E8', 'E9', 'E16', 'E17', 'E19', 'E20', 
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S8', 'S12', "I2", "P8", "P9", "P10"]
    
    # Create aggregate features
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    
    # Create advanced engineered features
    data = create_advanced_features(data)
    
    # Select columns - remove duplicates
    aggregate_features = ['M*', 'E*', 'I*', 'P*', 'V*', 'S*', 'D*']
    
    if typ == "train":
        # Get all feature columns (excluding target)
        all_feature_cols = [col for col in data.columns if col != 'forward_returns']
        # Remove duplicates while preserving order
        feature_cols_unique = []
        seen = set()
        for col in main_features + aggregate_features + all_feature_cols:
            if col not in seen and col in data.columns and col != 'forward_returns':
                feature_cols_unique.append(col)
                seen.add(col)
        
        data = data[feature_cols_unique + ['forward_returns']].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else:
        # Get all feature columns
        all_feature_cols = [col for col in data.columns]
        # Remove duplicates while preserving order
        feature_cols_unique = []
        seen = set()
        for col in main_features + aggregate_features + all_feature_cols:
            if col not in seen and col in data.columns:
                feature_cols_unique.append(col)
                seen.add(col)
        
        data = data[feature_cols_unique].copy()
    
    feature_cols = [col for col in data.columns if col != 'target']
    
    # Imputation
    if typ == "train":
        imputer = SimpleImputer(strategy='mean')
        data[feature_cols] = imputer.fit_transform(data[feature_cols])
    else:
        if imputer is None:
            raise ValueError("Imputer must be provided for test data")
        data[feature_cols] = imputer.transform(data[feature_cols])
    
    # Scaling
    if typ == "train":
        scaler = MinMaxScaler()
        data[feature_cols] = scaler.fit_transform(data[feature_cols])
    else:
        if scaler is None:
            raise ValueError("Scaler must be provided for test data")
        data[feature_cols] = scaler.transform(data[feature_cols])
    
    if typ == "train":
        return data, scaler, imputer
    else:
        return data


train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
train_processed, scaler, imputer = preprocessing(train, "train")

train_x = train_processed.drop(columns=["target"])
train_y = train_processed['target']

In [3]:
train_processed

,E1,E2,E3,E4,E5,E7,E8,E9,E16,E17,...,total_risk,signal_to_risk,macro_tech_ratio,sentiment_vol_ratio,price_interest_ratio,M_E_poly,V_S_poly,MEVIPS_mean,MEVIPS_std,target
6969,0.041781,0.727190,0.649542,0.021277,0.979815,0.894408,0.010199,0.018862,0.780305,0.733879,...,0.093642,0.110416,0.350922,0.407192,0.603714,0.924299,0.997628,0.420999,0.103510,0.001145
6970,0.041500,0.728277,0.650503,0.019640,0.980146,0.894352,0.010233,0.018531,0.780123,0.733868,...,0.119789,0.110320,0.372991,0.407295,0.604756,0.934085,0.998552,0.460149,0.095524,0.004738
6971,0.041219,0.736669,0.657605,0.018003,0.980477,0.894295,0.010266,0.018200,0.779941,0.733856,...,0.136870,0.110251,0.355208,0.407401,0.605065,0.912051,0.999076,0.468093,0.111625,0.006016
6972,0.040938,0.747184,0.666562,0.016367,0.980807,0.894394,0.011356,0.017869,0.776877,0.893087,...,0.144830,0.110200,0.357064,0.407552,0.604719,0.844999,0.999737,0.445780,0.136310,0.001414
6973,0.040657,0.750377,0.669411,0.014730,0.981138,0.888036,0.011387,0.017538,0.776701,0.892525,...,0.129612,0.110246,0.356494,0.407480,0.604600,0.871986,0.999538,0.441746,0.121810,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,0.282092,0.654241,0.781596,0.152209,0.331238,0.851819,0.042188,0.914626,0.627247,0.437709,...,0.086415,0.110459,0.361512,0.407615,0.605922,0.949038,0.999918,0.406126,0.061546,0.002457
8986,0.281430,0.656796,0.783496,0.150573,0.330907,0.851816,0.042187,0.914957,0.627242,0.437767,...,0.083120,0.110526,0.359627,0.408262,0.606286,0.973092,0.999250,0.440182,0.040120,0.002312
8987,0.280770,0.660234,0.786076,0.148936,0.330576,0.851813,0.042186,0.915288,0.627200,0.437777,...,0.090741,0.110388,0.359877,0.408261,0.605218,0.969171,0.999462,0.402175,0.041423,0.002891
8988,0.280112,0.664119,0.788992,0.147300,0.330245,0.851793,0.042186,0.915619,0.627159,0.437787,...,0.078195,0.110587,0.361216,0.407427,0.606791,0.955208,0.999712,0.395639,0.070944,0.008310


In [4]:
import numpy as np
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

class ResidualBoostingEnsemble:
    
    def __init__(self, base_params, n_models=3):
        self.base_params = base_params
        self.n_models = n_models
        self.models = []
        
    def fit(self, train_x, train_y, test_x=None, test_y=None):
        current_train_pred = np.zeros(len(train_y))
        
        for i in range(self.n_models):
            print(f'Training Model {i+1}...')
            
            if i == 0:
                target = train_y
            else:
                residuals = train_y - current_train_pred
                target = residuals
            
            model = LGBMRegressor(**self.base_params)
            model.fit(train_x, target)
            self.models.append(model)
            
            pred = model.predict(train_x)
            current_train_pred += pred
            
            train_mape = mean_absolute_percentage_error(train_y, current_train_pred)
            print(f'Cumulative Training MAPE: {train_mape:.4f}')
        
        return self
    
    def predict(self, X):
        predictions = np.zeros(len(X))
        for model in self.models:
            predictions += model.predict(X)
        return predictions
    
    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f'Model saved to {filepath}')
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)
    
    def evaluate(self, X, y):
        predictions = self.predict(X)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        print(f'MAPE: {mape:.4f}')
        print(f'MAE: {mae:.4f}')
        print(f'RMSE: {rmse:.4f}')
        
        return {'mape': mape, 'mae': mae, 'rmse': rmse}


LGBM_R_parm = {'boosting_type': 'gbdt', 
               'objective': 'regression_l1', 
               'metric': 'mape', 
               'colsample_bytree': 0.95, 
               'learning_rate': 0.08,
               'max_bin': 100,
               'max_depth': 12,
               'min_child_samples': 60,
               'min_data_in_leaf': 20, 
               'n_estimators': 7000,
               'num_leaves': 50,
               'reg_alpha': 0.8,
               'reg_lambda': 3.5, 
               'subsample': 0.75, 
               'verbosity': -1
              }

ensemble = ResidualBoostingEnsemble(base_params=LGBM_R_parm, n_models=3)
ensemble.fit(train_x, train_y)

ensemble.save('LGBM_Residual_Ensemble.pkl')

print('\nFinal Training Evaluation:')
ensemble.evaluate(train_x, train_y)

LGBM_R_model = joblib.load('LGBM_Residual_Ensemble.pkl')

Training Model 1...
Cumulative Training MAPE: 120330836.8319
Training Model 2...
Cumulative Training MAPE: 113514372.4719
Training Model 3...
Cumulative Training MAPE: 109905457.0656
Model saved to LGBM_Residual_Ensemble.pkl

Final Training Evaluation:
MAPE: 109905457.0656
MAE: 0.0001
RMSE: 0.0003


In [5]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)

def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        test_processed = preprocessing(test_df, 'inference', scaler, imputer)
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

Prediction error: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- is_scored
- lagged_forward_returns
- lagged_market_forward_excess_returns
- lagged_risk_free_rate
Feature names seen at fit time, yet now missing:
- market_forward_excess_returns
- risk_free_rate

Prediction error: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- is_scored
- lagged_forward_returns
- lagged_market_forward_excess_returns
- lagged_risk_free_rate
Feature names seen at fit time, yet now missing:
- market_forward_excess_returns
- risk_free_rate

Prediction error: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- is_scored
- lagged_forward_returns
- lagged_market_forward_excess_returns
- lagged_risk_free_rate
Feature names seen at fit time, yet now missing:
- market_forward_excess_returns
- risk_free_rate

Prediction error: The feature names should match 